In [1]:
import pandas as pd
import requests
import dotenv
import os
import json
from datetime import datetime
from urllib.parse import urljoin
import logging


In [2]:
#========================================================================================================================================================================
# Logger
#========================================================================================================================================================================
logger = logging.getLogger(__name__)

#========================================================================================================================================================================
# Carregando as variáveis do arquivo .env
#========================================================================================================================================================================
dotenv.load_dotenv(override=True)

api_url = os.getenv('API_URL')
api_username = os.getenv('API_USERNAME')
api_password = os.getenv('API_PASSWORD')


#========================================================================================================================================================================
# Variáveis globais
#========================================================================================================================================================================
timeout = 10


In [3]:

session_payload = {
    "nome": api_username,
    "password": api_password
}

auth_headers = {
    "Content-Type": "application/json"
}

session_url = urljoin(api_url, "sessions")
try:
    response = requests.request(
            method="POST",
            url=session_url,
            headers=auth_headers,
            json = session_payload, 
            timeout = timeout
        )
    
    response.raise_for_status() # Verifica se a resposta foi bem-sucedida (status code 200-299)
    
    #Recuperar o cookie de autenticação
    if response.status_code == 200:
        logger.info("Login bem-sucedido. Status code: 200")
        auth_cookie = response.cookies.get("userId")
        endpoint_headers={
        "cookie": f"userId={auth_cookie}"
    }
        if endpoint_headers['cookie']:
            print("Autenticação bem-sucedida. Cookie obtido.")
            print(endpoint_headers)

        else:
            print("Falha na autenticação. Verifique suas credenciais.")

    else:
        logger.warning(f"Login retornou status code inesperado: {response.status_code}")
    
    
    if not auth_cookie:
        logger.error("Cookie de autenticação não encontrado na resposta.")
       
    
    logger.info("Sessão iniciada com sucesso.") # Log de sucesso
    
    

except requests.exceptions.HTTPError as http_err:
    logger.error(f"Erro HTTP ao logar: {http_err}")
except Exception as err:
    logger.error(f"Erro inesperado no login: {err}")



Autenticação bem-sucedida. Cookie obtido.
{'cookie': 'userId=s%3A6d31sVKMVfShOe7zNz_aHkoudEnpigH6.TrvswOywbYwaw5cX4f%2BpHWZY9jnjwazD3KS2KbPy90w'}


# Pessoas

In [4]:
pessoas_url = f"{api_url}pessoas"
perfis_de_interesse = [
    "387",
    "388",
    "702"
]
pessoas_data = []
for perfil in perfis_de_interesse:
    
    try:
        query_params = {
            "nested": "true",
            "co_tipo_gn": perfil
        }
        response = requests.get(pessoas_url, headers=endpoint_headers, params=query_params, timeout=timeout)
        response.raise_for_status() # Verifica se a resposta foi bem-sucedida (status code 200-299)
        print(f"Perfil {perfil} - Status code: {response.status_code}")
        
        pessoas_data.append(pd.DataFrame(response.json()))
        
    except requests.exceptions.HTTPError as http_err:
        logger.error(f"Erro HTTP ao acessar endpoint de pessoas: {http_err}")
    except Exception as err:
        logger.error(f"Erro inesperado ao acessar endpoint de pessoas: {err}")
    
if pessoas_data:
    pessoas_filtrados_df = pd.concat(pessoas_data, ignore_index = True)
    print(f"Dataframe do perfil {perfil} criado com sucesso. Total de registros: {len(pessoas_filtrados_df)}")
else:
    logger.warning(f"Nenhum dado retornado para o perfil {perfil}. Dataframe não criado.")

Perfil 387 - Status code: 200
Perfil 388 - Status code: 200
Perfil 702 - Status code: 200
Dataframe do perfil 702 criado com sucesso. Total de registros: 794


In [5]:
pessoas_filtrados_df = pessoas_filtrados_df[[
    'id',
    'ds_nome',
    'dados_status',
    'dados_co_tipo_gn',
    'dados_centro',
    'dados_pessoa_especialidade'


]]

In [6]:
extrair_dados = [
    'dados_status',
    'dados_co_tipo_gn',
    'dados_centro',
    'dados_pessoa_especialidade'
]

for coluna in extrair_dados:
    dados_extraidos = pessoas_filtrados_df[coluna].apply(lambda x: x if isinstance(x, dict) else {})
    dados_expandido = pd.json_normalize(dados_extraidos)
    dados_expandido.columns = [f"{coluna}_{subcoluna}" for subcoluna in dados_expandido.columns]
    #excluir as colunas originais
    pessoas_filtrados_df = pessoas_filtrados_df.drop(columns=[coluna])
    pessoas_filtrados_df = pd.concat([pessoas_filtrados_df, dados_expandido], axis=1)

    

In [7]:
# 1. Limpeza inicial de espaços
pessoas_filtrados_df['ds_nome'] = pessoas_filtrados_df['ds_nome'].str.strip()

# 2. Regex Robusto
# ^(dr|dra|dras|drs) -> Pega as variações de doutor(a)
# [.\s]* -> Pega qualquer combinação de pontos ou espaços logo após
# (a\.|o\.)?         -> OPCIONAL: Pega um 'a.' ou 'o.' isolado que sobrou no início
# \s* -> Espaços extras
regex_final = r'^(dr|dra|dras|drs)[.\s]*(a\.|o\.)?\s*'

pessoas_filtrados_df['ds_nome'] = pessoas_filtrados_df['ds_nome'].str.replace(
    regex_final, '', regex=True, flags=2
)

# 3. Caso ainda existam iniciais soltas como "A. " ou "A " no início de tudo
# Isso limpa o "A. " que você mencionou que ficou no início.
pessoas_filtrados_df['ds_nome'] = pessoas_filtrados_df['ds_nome'].str.replace(
    r'^[AO][.\s]+', '', regex=True, flags=2
)

# 4. Strip final para garantir
pessoas_filtrados_df['ds_nome'] = pessoas_filtrados_df['ds_nome'].str.strip()

#+========================================

# 2. Criar uma coluna temporária para definir a prioridade
# Pesquisador Principal = 1 (Alta prioridade)
# Outros = 2 (Baixa prioridade)
prioridade = {
    'Pesquisador Principal': 1,
    'Subinvestigador': 2,
    'Equipe Médica': 3
}

# Criamos uma coluna de peso baseada no dicionário, se não encontrar o cargo, assume peso 4
pessoas_filtrados_df['prioridade_rank'] = pessoas_filtrados_df['dados_co_tipo_gn_ds_descricao'].map(prioridade).fillna(4)

# 3. Ordenar pela prioridade (do menor para o maior rank)
# Assim, o 1 (Pesquisador) fica sempre no topo do grupo de nomes duplicados
pessoas_filtrados_df = pessoas_filtrados_df.sort_values(by=['ds_nome', 'prioridade_rank'])

# 4. Remover duplicatas mantendo o primeiro (que agora é o de maior prioridade)
pessoas_filtrados_df = pessoas_filtrados_df.drop_duplicates(subset='ds_nome', keep='first')

# 5. Remover a coluna temporária de rank (opcional, para deixar o DF limpo)
pessoas_filtrados_df = pessoas_filtrados_df.drop(columns=['prioridade_rank'])

# Centros

In [8]:
centro_url = f"{api_url}centro"
try:
    query_params = {
        "nested": "true",
    }
    response = requests.get(centro_url, headers=endpoint_headers, timeout=timeout)
    response.raise_for_status() # Verifica se a resposta foi bem-sucedida (status code 200-299)
    
    centro_data = pd.DataFrame(response.json())
    
except requests.exceptions.HTTPError as http_err:
    logger.error(f"Erro HTTP ao acessar endpoint de centro: {http_err}")
except Exception as err:
    logger.error(f"Erro inesperado ao acessar endpoint de centro: {err}")